# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

Además de pandas/numpy/sklearn básicos, importamos varios modelos adicionales (Linear Regression, KNN, Decision Tree, XGBoost, LightGBM, CatBoost) y utilidades de validación cruzada, escalado y búsqueda de hiperparámetros que usaremos más adelante en la comparativa de modelos (sección 4).

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import  XGBRegressor
from lightgbm import  LGBMRegressor
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor

## 2. Datos

Cargamos `train.csv` con `encoding='latin-1'`: el CSV trae caracteres especiales (por ejemplo, símbolos dentro de la columna `Gpu`) que fallan al leerlos como UTF-8.

In [ ]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

Vistazo rápido a la tabla (`df.head()`), tipos y nulos (`df.info()` — no hay valores faltantes, y casi todas las columnas llegan como texto aunque describan números), y un repaso a las columnas más sucias: `ScreenResolution` mezcla resolución numérica con texto descriptivo (IPS Panel, Touchscreen...) en 36 valores distintos, y `Memory` mezcla texto y número, pudiendo llevar dos dispositivos en la misma celda (ej. `"128GB SSD + 1TB HDD"`). Cerramos con los estadísticos básicos de las columnas numéricas, incluyendo el target `Price_in_euros`.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
col = "ScreenResolution"
print(len(df[col].value_counts().index))
print(df[col].value_counts())

In [ ]:
col = "Memory"
print(len(df[col].value_counts().index))
print(df[col].value_counts())

In [ ]:
df.describe()

### 2.2 Definir X e y

Renombramos el target a `target` (nombre genérico): `test.csv` no trae `Price_in_euros`, así que el código de las secciones siguientes queda reutilizable para ambos. Después separamos en train/test con un split 80/20, fijando `random_state=42` para que el resultado sea reproducible entre ejecuciones.

In [ ]:
df.rename(columns={'Price_in_euros': 'target'}, inplace=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['target']), df['target'], test_size=0.2, random_state=42)

## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

### 3.1 Resolución de pantalla

De `ScreenResolution` extraemos el ancho/alto numérico (`Res_Width`, `Res_Height`) y unos flags booleanos (`Is_IPS_Panel`, `Is_Touchscreen`, `Is_FullHD`, `Is_QuadHD`, `Is_4K`, `Is_Retina`) a partir de las palabras clave del texto.

In [ ]:
def add_screen_features(df_in):
    df = df_in.copy()
    df[["Res_Width", "Res_Height"]] = df["ScreenResolution"].str.extract(r'(\d+)x(\d+)').astype(int)
    df["Is_IPS_Panel"]   = (df["ScreenResolution"].str.contains("IPS Panel", regex=False)).astype(int)
    df["Is_Touchscreen"] = (df["ScreenResolution"].str.contains("Touchscreen", regex=False)).astype(int)
    df["Is_FullHD"]      = (df["ScreenResolution"].str.contains("Full HD", regex=False)).astype(int)
    df["Is_QuadHD"]      = (df["ScreenResolution"].str.contains("Quad HD+", regex=False)).astype(int)
    df["Is_4K"]          = (df["ScreenResolution"].str.contains("4K Ultra HD", regex=False)).astype(int)
    df["Is_Retina"]      = (df["ScreenResolution"].str.contains("Retina Display", regex=False)).astype(int)
    return df

In [ ]:
X_train_screen = add_screen_features(X_train)
X_test_screen = add_screen_features(X_test)

### 3.2 Peso y RAM

`Weight` (`"1.86kg"`) y `Ram` (`"8GB"`) son texto con la unidad pegada al número. `remove_text_numeric_col` quita la unidad con regex y convierte a `float`, admitiendo decimales (`"1.86kg"` → `1.86`).

In [ ]:
def remove_text_numeric_col(df_in, col_name):
    df = df_in.copy()
    df[col_name] = df[col_name].str.extract(r'(\d+\.?\d*)').astype(float)
    return df

In [ ]:
X_train_weight = remove_text_numeric_col(X_train_screen, "Weight")
X_test_weight = remove_text_numeric_col(X_test_screen, "Weight")

X_train_ram = remove_text_numeric_col(X_train_weight, "Ram")
X_test_ram = remove_text_numeric_col(X_test_weight, "Ram")

### 3.3 Almacenamiento (`Memory`)

`Memory` puede tener uno o dos dispositivos en la misma celda (ej. `"128GB SSD + 1TB HDD"`) y mezcla GB con TB. `parse_row` separa la celda por el `"+"`, normaliza cada tamaño a GB (TB × 1000) y suma las cantidades cuando el mismo tipo de disco aparece dos veces en la misma celda; `add_memory_features` aplica eso fila a fila y crea `SSD_GB`, `HDD_GB`, `Flash_Storage_GB` y `Hybrid_GB`.

In [ ]:
def parse_row(cell):
    sizes = {"SSD": 0, "HDD": 0, "Flash Storage": 0, "Hybrid": 0}
    for part in re.split(r'\s*\+\s*', cell):
        amount, unit, mtype = re.match(
            r'(\d+\.?\d*)(GB|TB)\s*(SSD|HDD|Flash Storage|Hybrid)', part
        ).groups()
        amount = float(amount) * (1000 if unit == "TB" else 1)
        sizes[mtype] += amount
    return pd.Series(sizes)

def add_memory_features(df):
    sizes_df = df["Memory"].apply(parse_row)
    df["SSD_GB"] = sizes_df["SSD"]
    df["HDD_GB"] = sizes_df["HDD"]
    df["Flash_Storage_GB"] = sizes_df["Flash Storage"]
    df["Hybrid_GB"] = sizes_df["Hybrid"]
    return df

In [ ]:
X_train_memory = add_memory_features(X_train_ram)
X_test_memory = add_memory_features(X_test_ram)
X_train_memory.drop(columns=["Memory", "ScreenResolution"], inplace=True)
X_test_memory.drop(columns=["Memory", "ScreenResolution"], inplace=True)


### 3.4 CPU

De `Cpu` extraemos la familia (`Cpu_Family`: Core i3/i5/i7, Celeron, Ryzen...), la velocidad en GHz (`Cpu_GHz`, magnitud numérica comparable entre todas las CPUs) y la gama según la letra final del modelo Intel (`Cpu_Tier`). El mapeo de letra a gama es: `U` → `Bajo_Consumo` (baja potencia, típico de portátiles), `Y` → `Ultra_Bajo_Consumo` (un escalón por debajo de `U`, portátiles ultra-finos sin ventilador), `HQ`/`HK` → `Alto_Rendimiento` (alto rendimiento de 4 núcleos; el overclock de `HK` no influye en el precio de venta, así que se agrupan juntas), `M` → `Workstation_Movil` (Xeon móvil para estaciones de trabajo), y `Sin_Identificador` cuando el modelo no lleva letra (Celeron/Pentium/Atom con nomenclatura N/Z, o portátiles Apple que solo reportan el GHz). La detección usa dos patrones: la serie Y tiene la letra en medio del código (dígito + `Y` + 2 dígitos, ej. `"7Y75"`), mientras que el resto de letras van al final del número de modelo (ej. `"6700HQ"`, `"1505M"`). `Cpu_Family` y `Cpu_Tier` se codifican con One-Hot (fit solo en train).

In [ ]:
CPU_TIER_MAP = {
    "U":  "Bajo_Consumo",
    "Y":  "Ultra_Bajo_Consumo",
    "HQ": "Alto_Rendimiento",
    "HK": "Alto_Rendimiento",
    "M":  "Workstation_Movil",
}

def extract_cpu_tier(cpu):
    if re.search(r'\dY\d{2}', cpu):
        return CPU_TIER_MAP["Y"]
    match = re.search(r'\d{3,5}([A-Z]{1,2})(?:\s|$|V\d)', cpu)
    return CPU_TIER_MAP.get(match.group(1), "Sin_Identificador") if match else "Sin_Identificador"

def add_cpu_features(df):
    df["Cpu_Family"] = df["Cpu"].str.extract(
        r'(Core i[3579]|Core M|Celeron|Pentium|Atom|Xeon|A\d+-Series|E-Series|FX|Ryzen)'
    )
    df["Cpu_GHz"] = df["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)
    df["Cpu_Tier"] = df["Cpu"].apply(extract_cpu_tier)
    return df

In [ ]:
X_train_cpu = add_cpu_features(X_train_memory)
X_test_cpu = add_cpu_features(X_test_memory)

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train_cpu[["Cpu_Family", "Cpu_Tier"]])

cpu_ohe_cols = encoder.get_feature_names_out(["Cpu_Family", "Cpu_Tier"])

X_train_cpu[cpu_ohe_cols] = encoder.transform(X_train_cpu[["Cpu_Family", "Cpu_Tier"]])
X_test_cpu[cpu_ohe_cols]  = encoder.transform(X_test_cpu[["Cpu_Family", "Cpu_Tier"]])

X_train_cpu.drop(columns=["Cpu", "Cpu_Family", "Cpu_Tier"], inplace=True)
X_test_cpu.drop(columns=["Cpu", "Cpu_Family", "Cpu_Tier"], inplace=True)


### 3.5 GPU

A diferencia de la CPU, `Gpu` no tiene un equivalente a GHz: el número de modelo (`GTX 1050`, `Quadro M2000M`...) no es una magnitud comparable entre familias distintas. `classify_gpu` agrupa el texto en líneas reconocibles (GeForce GTX, GeForce MX, Quadro, Radeon, Radeon Pro, HD Graphics...), incluyendo el caso atípico `"Intel Graphics 620"` (sin el `HD` de en medio) dentro de `HD Graphics`. Dentro de `Radeon` (excluyendo Pro/RX) y de `GeForce GTX` el número sí es relevante para la capacidad de la gráfica, así que se desglosan en categorías propias: `Radeon R2`...`Radeon R7` según el número de gama, y `GeForce GTX 1050`, `GeForce GTX 1050 Ti`, `GeForce GTX 1060`... según el modelo, distinguiendo si lleva `Ti` (misma gama de número, pero la versión `Ti` es la más potente). El resultado se codifica con One-Hot (fit solo en train).

In [ ]:
def classify_gpu(value):
    radeon_r = re.search(r'\bR([2-7])\b', value)
    if radeon_r:
        return f"Radeon R{radeon_r.group(1)}"

    gtx = re.search(r'GTX\s?(\d{3,4})\s?(Ti)?', value)
    if gtx:
        modelo, ti = gtx.groups()
        return f"GeForce GTX {modelo}" + (" Ti" if ti else "")

    checks = [
        ("Radeon Pro", "Radeon Pro"),
        ("Radeon RX", "Radeon RX"),
        ("FirePro", "FirePro"),
        ("Radeon", "Radeon"),
        ("Quadro", "Quadro"),
        ("GTX", "GeForce GTX"),
        ("MX", "GeForce MX"),
        ("GeForce", "GeForce"),
        ("UHD Graphics", "UHD Graphics"),
        ("Iris Plus Graphics", "Iris Plus Graphics"),
        ("Iris Pro Graphics", "Iris Pro Graphics"),
        ("Iris Graphics", "Iris Graphics"),
        ("HD Graphics", "HD Graphics"),
        ("Graphics", "HD Graphics"),
    ]
    for keyword, label in checks:
        if keyword in value:
            return label
    return "Other"

In [ ]:
X_train_gpu = X_train_cpu.copy()
X_test_gpu = X_test_cpu.copy()
X_train_gpu["Gpu"] = X_train_cpu["Gpu"].apply(classify_gpu)
X_test_gpu["Gpu"] = X_test_cpu["Gpu"].apply(classify_gpu)

In [ ]:
X_train_gpu["Gpu"].value_counts()

In [ ]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train_gpu[["Gpu"]])

gpu_ohe_cols = encoder.get_feature_names_out(["Gpu"])
X_train_gpu[gpu_ohe_cols] = encoder.transform(X_train_gpu[["Gpu"]])
X_test_gpu[gpu_ohe_cols] = encoder.transform(X_test_gpu[["Gpu"]])
X_train_gpu.drop(columns=["Gpu"], inplace=True)
X_test_gpu.drop(columns=["Gpu"], inplace=True)

### 3.6 Unificamos en `X_train_cat` / `X_test_cat`

A partir de aquí trabajamos sobre una única tabla por split. `Product` en sí (480 valores únicos sobre 912 filas) es casi un identificador por fila — no generaliza y solo añadiría ruido/overfitting, así que lo quitamos. Pero antes extraemos `Product_Family`: la línea/serie del portátil (`ThinkPad`, `IdeaPad`, `ROG`, `Alienware`, `XPS`, `MacBook Pro`...), que sí generaliza y que ni `Company` ni `TypeName` capturan — por ejemplo, dentro de Lenovo, un `ThinkPad` cuesta de media más del doble que un `IdeaPad`, y ambos pueden compartir `TypeName`. `extract_product_family` recorre una lista de patrones (uno por línea/serie reconocible, agrupados por fabricante) y, si ninguno encaja, hace un último intento solo para MSI (cuyos modelos no tienen nombre de línea, solo un código de 2 letras + número, ej. `"GE62"` → serie `GE`) antes de caer en `"Other"`.

In [ ]:
PRODUCT_FAMILY_PATTERNS = [
    (r'MacBook Pro', 'MacBook Pro'),
    (r'MacBook Air', 'MacBook Air'),
    (r'MacBook', 'MacBook'),
    (r'Chromebook', 'Chromebook'),
    (r'Predator', 'Predator'),
    (r'Aspire', 'Aspire'),
    (r'Nitro', 'Nitro'),
    (r'Swift', 'Swift'),
    (r'Spin', 'Spin'),
    (r'TravelMate', 'TravelMate'),
    (r'Extensa', 'Extensa'),
    (r'ROG|Strix', 'ROG'),
    (r'ZenBook', 'ZenBook'),
    (r'VivoBook', 'VivoBook'),
    (r'Alienware', 'Alienware'),
    (r'Inspiron', 'Inspiron'),
    (r'Latitude', 'Latitude'),
    (r'Precision', 'Precision'),
    (r'Vostro', 'Vostro'),
    (r'XPS', 'XPS'),
    (r'EliteBook', 'EliteBook'),
    (r'ProBook', 'ProBook'),
    (r'Omen', 'Omen'),
    (r'Pavilion', 'Pavilion'),
    (r'Envy', 'Envy'),
    (r'Spectre', 'Spectre'),
    (r'Stream', 'Stream'),
    (r'ZBook', 'ZBook'),
    (r'ThinkPad', 'ThinkPad'),
    (r'IdeaPad', 'IdeaPad'),
    (r'Legion', 'Legion'),
    (r'Yoga', 'Yoga'),
    (r'Port[eé]g[eé]', 'Portege'),
    (r'Tecra', 'Tecra'),
    (r'Satellite', 'Satellite'),
    (r'Surface', 'Surface'),
    (r'Blade', 'Blade'),
    (r'LifeBook', 'LifeBook'),
]

def extract_product_family(product, company):
    for pattern, label in PRODUCT_FAMILY_PATTERNS:
        if re.search(pattern, product, flags=re.IGNORECASE):
            return label
    if company == "MSI":
        msi_match = re.match(r'([A-Z]{2})\d', product)
        if msi_match:
            return f"MSI {msi_match.group(1)}"
    return "Other"

In [ ]:
X_train_cat = X_train_gpu.copy()
X_test_cat = X_test_gpu.copy()
X_train_cat["Product_Family"] = X_train_cat.apply(lambda r: extract_product_family(r["Product"], r["Company"]), axis=1)
X_test_cat["Product_Family"] = X_test_cat.apply(lambda r: extract_product_family(r["Product"], r["Company"]), axis=1)
X_train_cat.drop(columns=["Product"], inplace=True)
X_test_cat.drop(columns=["Product"], inplace=True)
X_train_cat.head()


### 3.7 Agrupamos marcas raras (`Company`)

Las marcas con 2 o menos apariciones se agrupan en `"Other"` para no añadir ruido al encoding posterior. El umbral se calcula **solo sobre `X_train_cat`** y se aplica igual a `X_test_cat` (nunca se recalcula sobre test).

In [ ]:
rare_brands = X_train_cat["Company"].value_counts()[lambda x: x <= 2].index
X_train_cat["Company_grouped"] = X_train_cat["Company"].where(~X_train_cat["Company"].isin(rare_brands), "Other")
X_train_cat["Company_grouped"].value_counts()

X_test_cat["Company_grouped"] = X_test_cat["Company"].where(~X_test_cat["Company"].isin(rare_brands), "Other")
X_test_cat["Company_grouped"].value_counts()

### 3.8 Agrupamos sistemas operativos (`OpSys`)

`"Mac OS X"` y `"macOS"` son el mismo sistema operativo con dos nombres distintos (Apple renombró su SO), y `"Windows 10 S"` es una variante restringida de `"Windows 10"`. Los unificamos antes de codificar para no fragmentar la señal.

In [ ]:
X_train_cat["OpSys_grouped"] = X_train_cat["OpSys"].replace({"Mac OS X": "macOS", "Windows 10 S": "Windows 10"})
X_train_cat["OpSys_grouped"].value_counts()

X_test_cat["OpSys_grouped"] = X_test_cat["OpSys"].replace({"Mac OS X": "macOS", "Windows 10 S": "Windows 10"})
X_test_cat["OpSys_grouped"].value_counts()

### 3.9 One-Hot del resto de categóricas + índice

Codificamos `Company_grouped`, `TypeName`, `OpSys_grouped` y `Product_Family` con un encoder por columna (fit solo en train). Por último fijamos `laptop_ID` como índice de la tabla: ya no aporta como feature, pero lo necesitamos más adelante para emparejar las predicciones con cada fila en la submission.

In [ ]:
for col in ["Company_grouped", "TypeName", "OpSys_grouped", "Product_Family"]:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    encoder.fit(X_train_cat[[col]])

    cpu_ohe_cols = encoder.get_feature_names_out([col])

    X_train_cat[cpu_ohe_cols] = encoder.transform(X_train_cat[[col]])
    X_test_cat[cpu_ohe_cols]  = encoder.transform(X_test_cat[[col]])

X_train_cat.drop(columns=["Company","Company_grouped", "TypeName", "OpSys", "OpSys_grouped", "Product_Family"], inplace=True)
X_test_cat.drop(columns=["Company","Company_grouped", "TypeName", "OpSys", "OpSys_grouped", "Product_Family"], inplace=True)


In [ ]:
X_train_cat.set_index('laptop_ID', inplace=True)
X_test_cat.set_index('laptop_ID', inplace=True)
X_train_cat.head()

### 3.10 EDA de features antes de modelar

Antes de entrenar, revisamos qué variables aportan señal real: correlación con el precio, importancia según un Random Forest rápido, y qué columnas dummy (one-hot) tienen tan pocos casos positivos que son más ruido que información.

In [ ]:
corr_target = X_train_cat.apply(lambda col: np.corrcoef(col.values, y_train.values)[0, 1])
print("Top 15 features por correlacion absoluta con el precio:")
print(corr_target.reindex(corr_target.abs().sort_values(ascending=False).index).head(15))

rf_eda = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_eda.fit(X_train_cat, y_train)
importances = pd.Series(rf_eda.feature_importances_, index=X_train_cat.columns).sort_values(ascending=False)
print("\nTop 15 features por importancia (Random Forest):")
print(importances.head(15))

binary_cols = [c for c in X_train_cat.columns if X_train_cat[c].nunique() <= 2]
prevalence = X_train_cat[binary_cols].mean()
low_variance_cols = prevalence[(prevalence < 0.02) | (prevalence > 0.98)].index.tolist()
print(f"\n{len(low_variance_cols)} columnas dummy con prevalencia < 2% o > 98%:")
print(low_variance_cols)

### 3.11 ¿Eliminamos columnas de baja varianza? (análisis, no se aplica)

Las columnas dummy detectadas arriba tienen muy pocos casos positivos. En un primer intento las eliminamos, pero al medir el impacto real en el RMSE (sección 4) comprobamos que **empeora el resultado**: aunque son raras, identifican con precisión a portátiles atípicos (workstations, gama muy alta o muy baja), y al perderlas el modelo predice peor justo esos casos — que pesan mucho en el RMSE por el error al cuadrado. Probamos también umbrales más suaves que el de prevalencia <2%/>98% y el resultado empeoraba igualmente.

**Decisión: nos quedamos con las columnas completas.** Dejamos el análisis y el cálculo como referencia, sin aplicar ningún drop.

In [ ]:
print(f"Si elimináramos estas columnas pasaríamos de {X_train_cat.shape[1]} a {X_train_cat.shape[1] - len(low_variance_cols)} columnas.")
print("Decision: las mantenemos todas, X_train_cat y X_test_cat no cambian.")

### 3.12 ¿Seleccionamos features por importancia? (análisis, no se aplica)

Probamos `SelectFromModel` con varios umbrales (mediana, 0.1×media, 0.25×media, 0.5×media) y en **todos los casos el RMSE de XGBoost empeoraba** frente a usar las columnas completas — el patrón se repite con el mismo motivo que en 3.11: las columnas menos "importantes" en media siguen siendo decisivas para los outliers de precio.

**Decisión: no descartamos ninguna columna.** `X_train_final`/`X_test_final` son una copia íntegra de `X_train_cat`/`X_test_cat`; mostramos qué seleccionaría el modelo solo a título informativo, sin aplicar el filtro.

In [ ]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    threshold='median'
)
selector.fit(X_train_cat, y_train)

selected_features = X_train_cat.columns[selector.get_support()]
print(f"SelectFromModel marcaria como relevantes {len(selected_features)} de {X_train_cat.shape[1]} columnas:")
print(list(selected_features))

X_train_final = X_train_cat.copy()
X_test_final = X_test_cat.copy()

## 4. Modelado

### 4.1 Entrenamiento

Entrenamos un Random Forest simple como modelo baseline, sin afinar hiperparámetros todavía — sirve de referencia para medir cuánto mejoran los pasos siguientes.

In [ ]:
rnd_forest = RandomForestRegressor(max_depth=5, n_estimators=100, random_state=42)
rnd_forest.fit(X_train_final, y_train)

### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**. Calculamos el RMSE del baseline sobre el split de test local — esta cifra es la referencia contra la que comparamos las mejoras de la sección 4.3.

In [ ]:
predictions = rnd_forest.predict(X_test_final)
rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")


### 4.3 Optimización (up to you 🫰🏻)

#### Skew de las variables continuas y normalización antes del escalado

`StandardScaler` centra y reescala, pero no corrige asimetría: una variable muy sesgada sigue sesgada después de escalar, lo que penaliza a modelos sensibles a la forma de la distribución (KNN, Linear Regression). Medimos el skew de cada variable continua (excluyendo las columnas binarias/one-hot, donde el skew no tiene sentido) y probamos `log1p`, `sqrt` y `cbrt` para ver cuál lo reduce más.

Según esa tabla: `Inches` y `Cpu_GHz` no se tocan (ya están casi simétricas; log/sqrt/cbrt están pensadas para colas a la derecha y las empeorarían). `Ram`, `Weight`, `Res_Width`, `Res_Height` y `HDD_GB` se transforman con `log1p` (la que más reduce su skew). `SSD_GB` se transforma con `sqrt` (deja el skew prácticamente en 0). `Flash_Storage_GB` y `Hybrid_GB` tampoco se tocan: tienen 94%/99.5% de ceros (casi ningún portátil con ese tipo de disco), así que ninguna transformación las acerca a una distribución simétrica.

Para verlo de forma visual además de numérica, comparamos el histograma de cada variable transformada antes y después de aplicar `log1p`/`sqrt`.

In [ ]:
from scipy.stats import skew

continuous_cols = [c for c in ["Inches", "Ram", "Weight", "Res_Width", "Res_Height",
                                "SSD_GB", "HDD_GB", "Flash_Storage_GB", "Hybrid_GB", "Cpu_GHz"]
                   if c in X_train_final.columns]

print(f"{'columna':20s}{'original':>10s}{'log1p':>10s}{'sqrt':>10s}{'cbrt':>10s}   mejor")
for col in continuous_cols:
    vals = X_train_final[col].values
    s = {"ninguna": skew(vals), "log1p": skew(np.log1p(vals)), "sqrt": skew(np.sqrt(vals)), "cbrt": skew(np.cbrt(vals))}
    mejor = min(s, key=lambda k: abs(s[k]))
    print(f"{col:20s}{s['ninguna']:10.3f}{s['log1p']:10.3f}{s['sqrt']:10.3f}{s['cbrt']:10.3f}   {mejor}")

In [ ]:
log1p_cols = [c for c in ["Ram", "Weight", "Res_Width", "Res_Height", "HDD_GB"] if c in X_train_final.columns]
sqrt_cols  = [c for c in ["SSD_GB"] if c in X_train_final.columns]

X_train_final[log1p_cols] = np.log1p(X_train_final[log1p_cols])
X_test_final[log1p_cols]  = np.log1p(X_test_final[log1p_cols])

X_train_final[sqrt_cols] = np.sqrt(X_train_final[sqrt_cols])
X_test_final[sqrt_cols] = np.sqrt(X_test_final[sqrt_cols])

print("Transformaciones aplicadas:")
print("  log1p ->", log1p_cols)
print("  sqrt  ->", sqrt_cols)

In [ ]:
import matplotlib.pyplot as plt

cols_transformadas = [(c, "log1p") for c in log1p_cols] + [(c, "sqrt") for c in sqrt_cols]

fig, axes = plt.subplots(len(cols_transformadas), 2, figsize=(10, 3.2 * len(cols_transformadas)))

for row, (col, nombre_transf) in enumerate(cols_transformadas):
    axes[row, 0].hist(X_train_cat[col], bins=30, color='steelblue', edgecolor='black')
    axes[row, 0].set_title(f"{col} — antes")

    axes[row, 1].hist(X_train_final[col], bins=30, color='mediumseagreen', edgecolor='black')
    axes[row, 1].set_title(f"{col} — después de {nombre_transf}")

plt.tight_layout()
plt.show()

In [ ]:
X_train_cat_sc, X_test_cat_sc = X_train_final.copy(), X_test_final.copy()
scaler_reg = StandardScaler()
X_train_cat_sc = scaler_reg.fit_transform(X_train_cat_sc)
X_test_cat_sc = scaler_reg.transform(X_test_cat_sc)

#### Comparativa de modelos por validación cruzada

Antes de comparar, generamos una versión escalada de los datos (`StandardScaler`, fit solo en train) — la necesitan KNN y Linear Regression, sensibles a la escala de las variables; los modelos de árbol (Random Forest, XGBoost, LightGBM, CatBoost) no la usan. Con eso, probamos varios algoritmos con 5-fold CV, comparando por RMSE (la métrica de la competición) y visualizando el resultado en un gráfico de barras horizontales con el error ± std del CV. Usamos `scoring='neg_root_mean_squared_error'` porque `cross_val_score` siempre maximiza la métrica internamente — para una métrica de error como el RMSE, sklearn la expone en negativo, y por eso negamos el resultado (`-scores.mean()`) al mostrarlo.

In [ ]:
modelos_reg = {
    'KNN Regressor k=5  (baseline)     ': (KNeighborsRegressor(n_neighbors=5), X_train_cat_sc),
    'Decision Tree Regressor           ': (DecisionTreeRegressor(max_depth=5, random_state=42), X_train_cat),
    'Random Forest Regressor depth=5   ': (RandomForestRegressor(max_depth=5, n_estimators=100, random_state=42), X_train_cat),
    'Linear Regression                 ': (LinearRegression(), X_train_cat_sc),
    'XGBoost Regressor                 ': (XGBRegressor(max_depth=5, n_estimators=100, random_state=42), X_train_cat),
    'LightGBM Regressor                ': (LGBMRegressor(max_depth=5, n_estimators=100, random_state=42), X_train_cat),
    'CatBoost Regressor                ': (CatBoostRegressor(max_depth=5, n_estimators=100, random_state=42, verbose=0), X_train_cat),
}

In [ ]:
scores_reg = {}
print('Modelo                             RMSE Media   +-Std')
print('-' * 55)
for nombre, (modelo, X_datos) in modelos_reg.items():
    scores = cross_val_score(
        modelo, X_datos, y_train,
        cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    rmse_m = -scores.mean()
    rmse_s = scores.std()
    scores_reg[nombre] = (rmse_m, rmse_s, X_datos)
    print(f'{nombre}  {rmse_m:.4f}    {rmse_s:.4f}')

mejor_reg = min(scores_reg, key=lambda k: scores_reg[k][0])
print(f'\nMejor modelo: {mejor_reg.strip()}  RMSE={scores_reg[mejor_reg][0]:.4f}')


In [ ]:
import matplotlib.pyplot as plt
nombres_r = [n.strip() for n in scores_reg.keys()]
mapas_m   = [v[0] for v in scores_reg.values()]
mapas_s   = [v[1] for v in scores_reg.values()]
colores_r = ['gold' if n == mejor_reg.strip() else 'mediumseagreen' for n in nombres_r]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(nombres_r, mapas_m, xerr=mapas_s, color=colores_r, edgecolor='black', capsize=4)
ax.set_xlabel('RMSE (menor es mejor)')
ax.set_title('Comparacion de modelos de regresion (5-fold CV)')
ax.axvline(mapas_m[0], color='red', linestyle='--', linewidth=0.8, label='Baseline KNN')
ax.legend()
plt.tight_layout()
plt.show()

#### Optimización de hiperparámetros para el ganador (CatBoost)

CatBoost fue el modelo con menor RMSE en la comparativa anterior. `GridSearchCV` prueba todas las combinaciones de la grid (`depth`, `learning_rate`, `l2_leaf_reg`, `iterations`, `subsample`) por validación cruzada y se queda con la que da mejor RMSE medio; no necesita escalado al ser un modelo de árbol. Evaluamos el modelo ganador sobre el split de test local antes de reentrenar con todos los datos (sección 5).

In [ ]:
params_cat = {'depth': [3, 4, 6, 8],
          'learning_rate': [0.1, 0.2, 0.3, 0.4],
          'l2_leaf_reg': [1, 3, 5, 7],
          'iterations': [100, 250, 500, 750],
          'subsample': [0.6, 0.8, 1.0]
          }

xgb_model = CatBoostRegressor(random_state=42, verbose=False)
gs_reg = GridSearchCV(
    xgb_model, params_cat,
    cv=5, scoring='neg_root_mean_squared_error', verbose=0,
)
gs_reg.fit(X_train_cat, y_train)

print('Mejores hiperparametros CatBoost Regressor:')
print(gs_reg.best_params_)
print(f'Mejor RMSE (CV): {-gs_reg.best_score_:.4f}')

Mejores hiperparametros CatBoost Regressor:
{'depth': 4, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.3, 'subsample': 0.6}
Mejor RMSE (CV): 253.3549

In [ ]:
# # params_xgb = {
# #     'n_estimators':  [100, 200, 400],
# #     'max_depth':     [3, 5, 7],
# #     'learning_rate': [0.05, 0.1, 0.2],
# #     'subsample':     [0.8, 1.0],
# # }
# params_xgb = {
#     'n_estimators':  [100, 200, 400, 600, 800, 1000],
#     'max_depth':     [3, 4, 5, 6],
#     'learning_rate': [0.05, 0.1, 0.2],
#     'subsample':     [0.6,0.8, 1.0],
# }
# xgb_model = XGBRegressor(random_state=42)
# gs_reg = GridSearchCV(
#     xgb_model, params_xgb,
#     cv=5, scoring='neg_root_mean_squared_error', verbose=0, n_jobs=-1
# )
# gs_reg.fit(X_train_cat, y_train)   # XGBoost no necesita escalado

# print('Mejores hiperparametros XGBoost Regressor:')
# print(gs_reg.best_params_)
# print(f'Mejor RMSE (CV): {-gs_reg.best_score_:.4f}')

Mejores hiperparametros XGBoost Regressor:
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 400, 'subsample': 0.8}
Mejor RMSE (CV): 267.1398

In [ ]:
xgb_model = gs_reg.best_estimator_
predictions_xgb = xgb_model.predict(X_test_cat)
rmse = root_mean_squared_error(y_test, predictions_xgb)
print(f"RMSE: {rmse}")

## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

Para eso releemos el CSV completo (no el split de la sección 2): a partir de aquí, `X_data`/`y_data` pasan a ser el "train" definitivo.

In [ ]:
train_data = pd.read_csv('./data/train.csv', encoding='latin-1')
train_data.head()
X_data = train_data.drop(columns=['Price_in_euros']).copy()
y_data = train_data['Price_in_euros'].copy()
X_data.shape

### 5.1 Repetimos el procesado de la sección 3 sobre `X_data` completo

Repetimos exactamente el mismo procesado de la sección 3 (resolución → peso/RAM → memoria → CPU → GPU → resto de categóricas), pero ahora sobre `X_data` completo, sin split. La diferencia clave: cada encoder se vuelve a **ajustar** aquí, porque `X_data` es ahora el "train" definitivo, y se guarda en el diccionario `encoders` para poder reutilizarlo (solo con `.transform()`) sobre `test.csv` en la sección 7. También repetimos, a título informativo, el mismo análisis de columnas de baja varianza y `SelectFromModel` de las secciones 3.11-3.12 — sin aplicar ningún drop, por la misma razón que entonces.

In [ ]:
X_data_screen = add_screen_features(X_data)
X_data_screen.drop(columns="ScreenResolution", inplace=True)

X_data_weight = remove_text_numeric_col(X_data_screen, "Weight")
X_data_ram = remove_text_numeric_col(X_data_weight, "Ram")

X_data_memory = add_memory_features(X_data_ram)
X_data_memory.drop(columns="Memory", inplace=True)

X_data_cpu = add_cpu_features(X_data_memory)
encoders = {}
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_data_cpu[["Cpu_Family", "Cpu_Tier"]])
encoders["Cpu"] = encoder

cpu_ohe_cols = encoder.get_feature_names_out(["Cpu_Family", "Cpu_Tier"])

X_data_cpu[cpu_ohe_cols] = encoder.transform(X_data_cpu[["Cpu_Family", "Cpu_Tier"]])
X_data_cpu.drop(columns=["Cpu", "Cpu_Family", "Cpu_Tier"], inplace=True)

X_data_gpu = X_data_cpu.copy()
X_data_gpu["Gpu"] = X_data_cpu["Gpu"].apply(classify_gpu)

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_data_gpu[["Gpu"]])
encoders["Gpu"] = encoder

gpu_ohe_cols = encoder.get_feature_names_out(["Gpu"])
X_data_gpu[gpu_ohe_cols] = encoder.transform(X_data_gpu[["Gpu"]])
X_data_gpu.drop(columns=["Gpu"], inplace=True)

X_data_cat = X_data_gpu.copy()
rare_brands = X_data_cat["Company"].value_counts()[lambda x: x <= 2].index
encoders["rare_brands"] = rare_brands
X_data_cat["Company_grouped"] = X_data_cat["Company"].where(~X_data_cat["Company"].isin(rare_brands), "Other")
X_data_cat["OpSys_grouped"] = X_data_cat["OpSys"].replace({"Mac OS X": "macOS", "Windows 10 S": "Windows 10"})
X_data_cat["Product_Family"] = X_data_cat.apply(lambda r: extract_product_family(r["Product"], r["Company"]), axis=1)

for col in ["Company_grouped", "TypeName", "OpSys_grouped", "Product_Family"]:
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    enc.fit(X_data_cat[[col]])
    encoders[col] = enc

    ohe_cols = enc.get_feature_names_out([col])
    X_data_cat[ohe_cols] = enc.transform(X_data_cat[[col]])

X_data_cat.drop(columns=["Company","Company_grouped", "TypeName", "OpSys", "OpSys_grouped", "Product", "Product_Family"], inplace=True)
X_data_cat.set_index('laptop_ID', inplace=True)

X_data_cat.head()


In [ ]:
binary_cols_data = [c for c in X_data_cat.columns if X_data_cat[c].nunique() <= 2]
prevalence_data = X_data_cat[binary_cols_data].mean()
low_variance_cols_data = prevalence_data[(prevalence_data < 0.02) | (prevalence_data > 0.98)].index.tolist()
print(f"Columnas de baja varianza detectadas (informativo, no se eliminan): {len(low_variance_cols_data)}")

selector_data = SelectFromModel(
    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    threshold='median'
)
selector_data.fit(X_data_cat, y_data)
selected_features_data = X_data_cat.columns[selector_data.get_support()]
print(f"SelectFromModel marcaria como relevantes (informativo, no se descartan las demas): {len(selected_features_data)} de {X_data_cat.shape[1]}")

print(f"\nDecision (igual que en la seccion 3): mantenemos las {X_data_cat.shape[1]} columnas completas de X_data_cat.")

### 5.2 Entrenamos el modelo final

Entrenamos el modelo ganador con los mismos hiperparámetros que obtuvo el GridSearch de la sección 4, ahora sobre el 100% de los datos (no necesita escalado, al ser un modelo de árbol). Por último, un chequeo rápido: predicciones sobre los propios datos de entrenamiento, que no se usan para nada más adelante — solo para confirmar que el modelo entrenado funciona antes de pasar a `test.csv`.

In [ ]:
xgb_model.fit(X_data_cat, y_data)

In [ ]:
y_pred = xgb_model.predict(X_data_cat)

---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

`test.csv` tiene las mismas columnas que `train.csv` excepto `Price_in_euros` — esa es justamente la columna que hay que predecir.

In [ ]:
X_data_test = pd.read_csv('./data/test.csv', encoding='latin-1')
X_data_test.shape

## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

Repetimos el mismo procesado de las secciones 3 y 5 (resolución → peso/RAM → memoria → CPU → GPU → resto de categóricas), pero aquí **solo transformamos**: todos los encoders ya estaban ajustados sobre `X_data` en la sección 5, y los recuperamos del diccionario `encoders`. En ningún momento se vuelve a llamar a `.fit()`.

In [ ]:
X_data_test_screen = add_screen_features(X_data_test)
X_data_test_screen.drop(columns="ScreenResolution", inplace=True)

X_data_test_weight = remove_text_numeric_col(X_data_test_screen, "Weight")
X_data_test_ram = remove_text_numeric_col(X_data_test_weight, "Ram")

X_data_test_memory = add_memory_features(X_data_test_ram)
X_data_test_memory.drop(columns="Memory", inplace=True)

X_data_test_cpu = add_cpu_features(X_data_test_memory)
encoder = encoders["Cpu"]
cpu_ohe_cols = encoder.get_feature_names_out(["Cpu_Family", "Cpu_Tier"])

X_data_test_cpu[cpu_ohe_cols] = encoder.transform(X_data_test_cpu[["Cpu_Family", "Cpu_Tier"]])
X_data_test_cpu.drop(columns=["Cpu", "Cpu_Family", "Cpu_Tier"], inplace=True)

X_data_test_gpu = X_data_test_cpu.copy()
X_data_test_gpu["Gpu"] = X_data_test_cpu["Gpu"].apply(classify_gpu)

encoder = encoders["Gpu"]
gpu_ohe_cols = encoder.get_feature_names_out(["Gpu"])
X_data_test_gpu[gpu_ohe_cols] = encoder.transform(X_data_test_gpu[["Gpu"]])
X_data_test_gpu.drop(columns=["Gpu"], inplace=True)

X_data_test_cat = X_data_test_gpu.copy()
rare_brands = encoders["rare_brands"]
X_data_test_cat["Company_grouped"] = X_data_test_cat["Company"].where(~X_data_test_cat["Company"].isin(rare_brands), "Other")
X_data_test_cat["OpSys_grouped"] = X_data_test_cat["OpSys"].replace({"Mac OS X": "macOS", "Windows 10 S": "Windows 10"})
X_data_test_cat["Product_Family"] = X_data_test_cat.apply(lambda r: extract_product_family(r["Product"], r["Company"]), axis=1)

for col in ["Company_grouped", "TypeName", "OpSys_grouped", "Product_Family"]:
    encoder = encoders[col]
    cpu_ohe_cols = encoder.get_feature_names_out([col])
    X_data_test_cat[cpu_ohe_cols] = encoder.transform(X_data_test_cat[[col]])

X_data_test_cat.drop(columns=["Company","Company_grouped", "TypeName", "OpSys", "OpSys_grouped", "Product", "Product_Family"], inplace=True)
X_data_test_cat.set_index('laptop_ID', inplace=True)

X_data_test_cat.head()

In [ ]:
pred_submit = xgb_model.predict(X_data_test_cat)

## 8. Genera la submission

Con el modelo ya reentrenado y el `test.csv` procesado, generamos las predicciones finales (`pred_submit`) y las convertimos en el archivo que se sube a Kaggle.

### 8.1 ¿Qué formato espera Kaggle?

`sample_submission.csv` define el formato exacto que espera Kaggle: columnas `laptop_ID` + `Price_in_euros`, en el mismo orden de filas que `test.csv`.

In [ ]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

### 8.2 Crea tu submission

Construimos la submission con los mismos `laptop_ID` y el mismo orden que `sample_submission`, emparejados con nuestras predicciones.

In [ ]:
sample_submission = pd.read_csv('./data/sample_submission.csv')
sample_submission.head()
ids = sample_submission["laptop_ID"]
primer_submit = pd.DataFrame({"laptop_ID": ids, "Price_in_euros": pred_submit})
primer_submit.head()

### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [ ]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [ ]:
checker(primer_submit, sample_submission, filename='submission_XGB_completo.csv')